

---

### 🚀 ขั้นตอนที่ 1: ติดตั้ง Library ขั้นสูง

นำโค้ดนี้ไปรันใน Cell แรกสุดของ Colab (เราจะใช้โมเดลระดับ State-of-the-Art ของ BAAI ที่เก่งภาษาไทยมาก)

In [3]:
# Load the data from kaggle and upload here.
!unzip data.zip

Archive:  data.zip
   creating: data/
   creating: data/knowledge_base/
   creating: data/knowledge_base/policies/
  inflating: data/knowledge_base/policies/cancellation_policy.md  
  inflating: data/knowledge_base/policies/membership_points_policy.md  
  inflating: data/knowledge_base/policies/return_policy.md  
  inflating: data/knowledge_base/policies/shipping_policy.md  
  inflating: data/knowledge_base/policies/warranty_policy.md  
   creating: data/knowledge_base/products/
  inflating: data/knowledge_base/products/AW-MN-001_arcwave_proview_27_4k.md  
  inflating: data/knowledge_base/products/AW-SK-001_arcwave_soundpillar_300.md  
  inflating: data/knowledge_base/products/DN-DT-001_daonuea_tower_x10.md  
  inflating: data/knowledge_base/products/DN-DT-002_daonuea_tower_x10_max.md  
  inflating: data/knowledge_base/products/DN-DT-003_daonuea_mini_pc_m1.md  
  inflating: data/knowledge_base/products/DN-DT-004_daonuea_all_in_one_27.md  
  inflating: data/knowledge_base/products/DN-DT

In [4]:
!pip install -q sentence-transformers pythainlp rank-bm25 requests python-dotenv

---

### 🧠 ขั้นตอนที่ 2: The Ultimate Retrieval & Reranking Pipeline

ก๊อปปี้โค้ดทั้งหมดนี้ไปวางใน Cell ใหม่ (อย่าลืมใส่ API Key ของคุณในตัวแปร `THAILLM_API_KEY`) โค้ดนี้จะรวมตั้งแต่การทำ Hybrid Search ไปจนถึงการให้คะแนนใหม่ด้วย Cross-Encoder

In [ ]:
import os, csv, re, time, requests
import numpy as np
from pathlib import Path
from pythainlp.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

# ==========================================
# 1. Configuration & API Setup
# ==========================================
N_QUESTIONS = 100 # รันจริง 100 ข้อ
DATA_DIR = "/content/data"
KB_DIR = f"{DATA_DIR}/knowledge_base"
THAILLM_API_KEY = '7HPY7sWIXQpzc2UcBM7Tee18QclsvbJW' # ใส่ Key ของคุณ

def ask_llm(messages, model="openthaigpt", max_retries=5):
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 1024,
        "temperature": 0.0, # บังคับให้ AI ตอบตามบริบทเป๊ะๆ ไม่มีความคิดสร้างสรรค์
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)
            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()
        except requests.exceptions.RequestException as e:
            time.sleep(2 ** attempt)
    return None

def parse_answer(text):
    if text is None: return 1
    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    m = re.search(r"ANSWER:\s*(\d+)", clean)
    if m: return int(m.group(1))
    for d in re.findall(r"\b(\d{1,2})\b", clean):
        if 1 <= int(d) <= 10: return int(d)
    return 1

# ==========================================
# 2. Load & Advanced Chunking
# ==========================================
CHUNK_SIZE = 800
CHUNK_OVERLAP = 200

kb_dir = Path(KB_DIR)
chunks = []
for fp in sorted(kb_dir.rglob("*.md")):
    text = fp.read_text(encoding="utf-8")
    category = fp.parent.name # ดึงชื่อโฟลเดอร์มาเป็นหมวดหมู่

    # Text split
    start = 0
    while start < len(text):
        chunk_text = text[start : start + CHUNK_SIZE]
        # ใส่ Metadata (ชื่อหมวดและไฟล์) เข้าไปในเนื้อหาเลยเพื่อให้ BM25 และ Dense จับได้
        enriched_text = f"[หมวดหมู่: {category}, ไฟล์: {fp.name}]\n{chunk_text}"
        chunks.append({"text": enriched_text, "source": str(fp.relative_to(kb_dir))})
        start += CHUNK_SIZE - CHUNK_OVERLAP

print(f"Created {len(chunks)} enriched chunks.")

# ==========================================
# 3. Initialize Models (Dense, Sparse, Reranker)
# ==========================================
print("Loading Models...")
# 3.1 Dense Model (BAAI/bge-m3 สำหรับหลายภาษา)
embed_model = SentenceTransformer("BAAI/bge-m3")
chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embed_model.encode(chunk_texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)

# 3.2 Sparse Model (BM25)
tokenized_chunks = [word_tokenize(c["text"], engine="newmm") for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

# 3.3 Reranker Model (Cross-Encoder)
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

# ==========================================
# 4. Ultimate Retrieval Pipeline
# ==========================================
def retrieve_and_rerank(query, top_k_final=5, fetch_k=20):
    # Step A: Dense Retrieval
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    dense_scores = np.dot(chunk_embeddings, q_emb.T).flatten()
    dense_idx = np.argsort(dense_scores)[::-1][:fetch_k]

    # Step B: Sparse Retrieval (BM25)
    tokens = word_tokenize(query, engine="newmm")
    bm25_scores = bm25.get_scores(tokens)
    bm25_idx = np.argsort(bm25_scores)[::-1][:fetch_k]

    # Step C: Hybrid Fusion (RRF)
    rrf_scores = {}
    for rank, idx in enumerate(dense_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (60 + rank)
    for rank, idx in enumerate(bm25_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (60 + rank)

    # เอา Top 15 จาก Hybrid ไปส่งให้ Cross-Encoder
    hybrid_top_idx = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:15]
    candidate_chunks = [chunks[i] for i in hybrid_top_idx]

    # Step D: Cross-Encoder Reranking
    pairs = [[query, chunk["text"]] for chunk in candidate_chunks]
    rerank_scores = reranker.predict(pairs)

    # จับคู่ Chunk กับคะแนนใหม่ แล้วเรียงลำดับ
    scored_candidates = list(zip(candidate_chunks, rerank_scores))
    scored_candidates.sort(key=lambda x: x[1], reverse=True)

    # คืนค่าเฉพาะ Top K ที่คะแนนสูงสุด
    return [chunk for chunk, score in scored_candidates[:top_k_final]]

---

### 📝 ขั้นตอนที่ 3: รันสร้าง Submission (Colab Ready)

โค้ดส่วนนี้จะอ่านคำถามทั้งหมด สร้าง Prompt แบบเข้มงวดเพื่อจัดการ Choice 9 และ 10 แล้วดึงคำตอบลงไฟล์ `submission.csv` ทันที

In [ ]:
# ==========================================
# 5. Generate Submission
# ==========================================
SYSTEM_PROMPT = """คุณคือ AI ผู้ช่วยบริการลูกค้าของร้าน FahMai (ฟ้าใหม่) ซึ่งเป็นร้านขายอุปกรณ์อิเล็กทรอนิกส์ มือถือ คอมพิวเตอร์ และเครื่องใช้ไฟฟ้า

หน้าที่ของคุณคือเลือกตัวเลือก 1-10 ที่ถูกต้องที่สุด โดยใช้ "ลำดับการคิด" ดังต่อไปนี้อย่างเคร่งครัด:

ขั้นที่ 1: วิเคราะห์หมวดหมู่คำถาม (Domain Check)
- คำถามนี้เกี่ยวกับ "อุปกรณ์ไอที, เครื่องใช้ไฟฟ้า, แบรนด์สินค้า, นโยบายของร้าน FahMai หรือการซื้อขาย" หรือไม่?
- หากคำถามเป็นเรื่องนอกหมวดหมู่โดยสิ้นเชิง (เช่น สูตรอาหาร, สภาพอากาศ, ตั๋วเครื่องบิน, ดอกเบี้ยธนาคาร, วันหยุดราชการ, มุกตลก) -> บังคับให้ตอบ "10" (คำถามนี้ไม่เกี่ยวข้อง) ทันที ห้ามไปขั้นต่อไป

ขั้นที่ 2: ค้นหาข้อมูล (Knowledge Check)
- หากคำถาม "เกี่ยวข้อง" (ผ่านขั้นที่ 1 มาได้) ให้มาค้นดูใน Context ที่ให้ไป
- หากพบข้อมูลที่ตอบคำถามได้อย่างชัดเจน -> เลือกคำตอบข้อ "1-8" ที่ตรงที่สุด
- หากคำถามเกี่ยวกับสินค้า/ร้าน แต่ใน Context "ไม่ได้ระบุเนื้อหานี้เอาไว้" หรือ "ข้อมูลไม่เพียงพอ" -> บังคับให้ตอบ "9" (ไม่มีข้อมูลนี้ในฐานข้อมูล)

กฎเหล็ก:
- แยก 9 กับ 10 ให้ออก: 9 = ถามเรื่องร้านแต่หาข้อมูลไม่เจอ, 10 = ถามเรื่องนอกเรื่องที่ไม่เกี่ยวกับร้านไอทีเลย
- ห้ามคิดคำตอบเอาเองถ้าไม่มีใน Context
- ตอบในรูปแบบ ANSWER: X (โดย X คือเลข 1-10 เท่านั้น)"""

# Load Questions
questions = []
with open(f"{DATA_DIR}/questions.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
        questions.append({"id": int(row["id"]), "question": row["question"], "choices": choices})

print("Starting Inference...")
predictions = {}

for q in questions[:N_QUESTIONS]:
    # 1. ค้นหาเอกสารด้วยสุดยอด Pipeline
    retrieved_chunks = retrieve_and_rerank(q["question"], top_k_final=5)

    context = "\n\n".join(f"--- Document ---\n{c['text']}" for c in retrieved_chunks)
    choices_text = "\n".join(f"{k}. {v}" for k, v in q["choices"].items())

    prompt = (
        f"Context:\n{context}\n\n"
        f"Question: {q['question']}\n\n"
        f"Choices:\n{choices_text}\n\n"
        f"ตอบ ANSWER: X"
    )

    # 2. ถาม LLM
    raw_answer = ask_llm([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ])

    # 3. แปลงคำตอบและเก็บผลลัพธ์
    pred = parse_answer(raw_answer)
    predictions[q["id"]] = pred
    print(f"Q{q['id']:>3}: Pred = {pred}")

# เขียนไฟล์ Submission
with open("submission_pathumma.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer"])
    for q in questions[:N_QUESTIONS]:
        writer.writerow([q["id"], predictions.get(q["id"], 1)])

print("\n✅ Done! Saved to submission_pathumma.csv")